# Phần 1: Câu hỏi Trắc nghiệm

In [1]:
import pandas as pd
import numpy as np

### Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

### Ý tưởng:
- Gom tất cả giao dịch của cùng một người vào với nhau rồi sắp xếp theo thứ tự thời gian
- Tính toán khoảng cách thời gian giữa các lần mua hàng
- Loại bỏ giá trị của các lần mua đầu tiên
- Tính trung vị

In [3]:
df = pd.read_csv("data/orders.csv")
df['order_date'] = pd.to_datetime(df['order_date'])
df = df.sort_values(by=['customer_id', 'order_date'])
df['inter_order_gap'] = df.groupby('customer_id')['order_date'].diff().dt.days
gaps = df['inter_order_gap'].dropna()
median_gap = gaps.median()
print(f"Trung vị số ngày giữa hai lần mua liên tiếp là: {median_gap} ngày")

Trung vị số ngày giữa hai lần mua liên tiếp là: 144.0 ngày


### Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?

### Ý tưởng:
- X = (price-cogs)/price
- Nhóm các sản phẩm lại theo cột segment và tính giá trị trung bình của X cho mỗi nhóm

In [6]:
df2 = pd.read_csv('data/products.csv')
df2['margin'] = (df2['price'] - df2['cogs']) / df2['price']
segment_ranking = df2.groupby('segment')['margin'].mean().sort_values(ascending=False)
print(f'Tỷ suất lợi nhuận gộp trung bình theo phân khúc: {segment_ranking}')

Tỷ suất lợi nhuận gộp trung bình theo phân khúc: segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: margin, dtype: float64


### Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

### Ý tưởng:
- Lọc file products.csv để lấy các sản phẩm Streetwear
- Sử dụng hàm merge để kết nối bảng trả hàng (returns.csv) với danh sách sản phẩm Streetwear vừa lọc thông qua mã sản phẩm (product_id).
- Đếm số lần xuất hiện của từng lý do trong cột return_reason của bảng sau khi đã kết hợp.

In [7]:
returns_df = pd.read_csv('data/returns.csv')
products_df = pd.read_csv('data/products.csv')
streetwear_products = products_df[products_df['category'] == 'Streetwear']
joined_data = pd.merge(returns_df, streetwear_products, on='product_id')
reason_counts = joined_data['return_reason'].value_counts()
print(f"Thống kê lý do trả hàng cho danh mục Streetwear: {reason_counts}")

Thống kê lý do trả hàng cho danh mục Streetwear: return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


### Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

### Ý tưởng:
- Tập hợp tất cả các bản ghi theo cột traffic_source.
- Tính giá trị trung bình của cột bounce_rate cho từng nhóm.
- Sắp xếp các kết quả tăng dần để tìm ra nguồn có con số nhỏ nhất

In [8]:
df3 = pd.read_csv('data/web_traffic.csv')
bounce_rate_stats = df3.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print(f"Tỷ lệ thoát trung bình theo từng nguồn truy cập: {bounce_rate_stats}")

Tỷ lệ thoát trung bình theo từng nguồn truy cập: traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64


### Q5. Tỷ lệ phần dtrăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

### Ý tưởng:
- Chỉ đếm các dòng mà giá trị ở cột promo_id không bị trống (not null).
- Lấy số lượng có khuyến mãi chia cho tổng số lượng và nhân với 100.

In [16]:
df4 = pd.read_csv('data/order_items.csv', low_memory=False)
total_rows = len(df4)
promo_rows = df4['promo_id'].notnull().sum()
percentage = (promo_rows / total_rows) * 100
print(f"Tỷ lệ phần trăm: {percentage:.2f}%")

Tỷ lệ phần trăm: 38.66%


### Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

### Ý tưởng:
- Gán thông tin age_group từ bảng khách hàng vào từng đơn hàng trong bảng orders.csv thông qua mã định danh customer_id.
- Loại bỏ các trường hợp không có thông tin nhóm tuổi (null).
- Đếm tổng số bản ghi đơn hàng trong mỗi nhóm
- Đếm số lượng khách hàng duy nhất (unique) thuộc nhóm tuổi đó trong hệ thống.
- Chia tổng số đơn cho tổng số khách hàng để tìm con số trung bình.

In [17]:
df_customers = pd.read_csv('data/customers.csv')
df_orders = pd.read_csv('data/orders.csv')
merged_df = pd.merge(df_orders, df_customers[['customer_id', 'age_group']], on='customer_id', how='left')
merged_df = merged_df[merged_df['age_group'].notnull()]
total_orders = merged_df.groupby('age_group')['order_id'].count()
total_customers = df_customers[df_customers['age_group'].notnull()].groupby('age_group')['customer_id'].nunique()
avg_orders = (total_orders / total_customers).sort_values(ascending=False)

print(f"Số đơn hàng trung bình trên mỗi khách hàng theo nhóm tuổi: {avg_orders}")

Số đơn hàng trung bình trên mỗi khách hàng theo nhóm tuổi: age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
dtype: float64


### Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất?

### Ý tưởng:
- Trong order_items.csv, tính doanh thu cho từng dòng bằng công thức: Revenue = (Quantity * UnitPrice) - DiscountAmount.
- Tổng hợp lại theo từng mã đơn hàng (order_id).
- Kết nối bảng doanh thu trên với orders.csv để biết mỗi đơn hàng được đặt ở mã bưu điện (zip) nào.
- Kết nối tiếp với geography.csv để biết mã bưu điện đó thuộc vùng (region) nào.
- Cộng tổng doanh thu của tất cả các đơn hàng theo từng vùng và tìm vùng có con số lớn nhất.

In [22]:
orders = pd.read_csv('data/orders.csv')
order_items = pd.read_csv('data/order_items.csv', low_memory=False)
geography = pd.read_csv('data/geography.csv')

order_items['line_revenue'] = (order_items['quantity'] * order_items['unit_price']) - order_items['discount_amount']
order_revenue = order_items.groupby('order_id')['line_revenue'].sum().reset_index()
order_with_zip = pd.merge(order_revenue, orders[['order_id', 'zip']], on='order_id')
final_data = pd.merge(order_with_zip, geography[['zip', 'region']], on='zip')
region_ranking = final_data.groupby('region')['line_revenue'].sum().sort_values(ascending=False)

print(f"Xếp hạng tổng doanh thu theo vùng: {region_ranking}")

Xếp hạng tổng doanh thu theo vùng: region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: line_revenue, dtype: float64


### Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

### Ý tưởng:
- Chỉ giữ lại những dòng có order_status chính xác là 'cancelled'.
- Trong nhóm các đơn hàng đã lọc, tiến hành đếm số lần xuất hiện của từng loại hình thanh toán trong cột payment_method.
- So sánh các con số vừa đếm được

In [24]:
df5 = pd.read_csv('data/orders.csv')
cancelled_df = df5[df_orders['order_status'] == 'cancelled']
payment_stats = cancelled_df['payment_method'].value_counts()
most_common = payment_stats.idxmax()
count = payment_stats.max()
print(f"Phương thức thanh toán bị hủy nhiều nhất là: {most_common} với {count} đơn hàng.")

Phương thức thanh toán bị hủy nhiều nhất là: credit_card với 28452 đơn hàng.


### Q8. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?

### Ý tưởng:
- Kết nối cả hai bảng order_items và returns với bảng products thông qua mã sản phẩm (product_id) để biết mỗi dòng dữ liệu tương ứng với size nào (S, M, L, hay XL).
- Tính tổng số dòng sản phẩm đã được bán ra (trong order_items) cho từng loại kích thước.
- Tính tổng số bản ghi trả hàng (trong returns) cho từng loại kích thước.
- Lấy (Số bản ghi trả hàng) chia cho (Số dòng đơn hàng đã bán) của từng size.

In [26]:
order_items = pd.read_csv('data/order_items.csv', low_memory = False)
returns = pd.read_csv('data/returns.csv')
products = pd.read_csv('data/products.csv')

orders_with_size = pd.merge(order_items, products[['product_id', 'size']], on='product_id')
order_counts = orders_with_size.groupby('size').size()

returns_with_size = pd.merge(returns, products[['product_id', 'size']], on='product_id')
return_counts = returns_with_size.groupby('size').size()

return_rate = (return_counts / order_counts).sort_values(ascending=False)

print(f"Tỷ lệ trả hàng theo kích thước sản phẩm: {return_rate}")

Tỷ lệ trả hàng theo kích thước sản phẩm: size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64


### Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

### Ý tưởng:
- Sử dụng cột installments (số kỳ trả góp) làm tiêu chí phân nhóm và cột payment_value (giá trị thanh toán) để tính toán.
- Nhóm tất cả các giao dịch theo số kỳ trả góp
- Với mỗi nhóm, tính giá trị trung bình của các khoản thanh toán để loại bỏ sự khác biệt về số lượng đơn hàng giữa các nhóm để thấy được nhóm nào thường có đơn hàng giá trị cao.

In [27]:
df_payments = pd.read_csv('data/payments.csv')
installment_ranking = df_payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print(f"Giá trị thanh toán trung bình theo kế hoạch trả góp: {installment_ranking}")

Giá trị thanh toán trung bình theo kế hoạch trả góp: installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64


# Kết luận:
### Đáp án trắc nghiệm là:
Q1: 144 ngày
Q2: Standard
Q3: wrong_size
Q4: email_campaign
Q5: 39%
Q6: 55+
Q7: East
Q8: credit_card
Q9: S
Q10: 6 kỳ